# SeisMambaKAN — Production training on Colab A100

Pre-flight (already baked into the configs):
- Phase 3 architecture (wider channels, sparse KAN, stem skip, beefier Mamba)
- Phase 4 training recipe (bf16 autocast, EMA decay=0.999, OneCycleLR, batch=128, epochs=100)
- Resume-from-checkpoint enabled (`--resume <exp_id>`) for Colab-disconnect recovery

Cell order:
1. Bootstrap (Drive mount + repo clone/pull)
2. Setup (deps + data sync — first run only)
3. Status snapshot
4. **Production train** (full STEAD, A100)
5. Resume (if Colab dropped mid-run)
6. Eval / Infer / TensorBoard

Open directly via:  
`https://colab.research.google.com/github/huseyinokanozturk/SeisMambaKAN/blob/main/notebooks/Colab.ipynb`

## 1) Bootstrap — Drive + GitHub clone

In [ ]:
# Mount Drive and clone (or pull) the GitHub repo.
import os, subprocess
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/SeisMambaKAN'
REPO_URL = 'https://github.com/huseyinokanozturk/SeisMambaKAN.git'

if Path(REPO_DIR, '.git').exists():
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
import sys; sys.path.insert(0, REPO_DIR)
print('cwd =', os.getcwd())

## 2) Setup — deps + data sync (first run only)

`--data-mode all` syncs the full STEAD shards from Drive (~90 GB; takes a while the first time, cached afterwards). For a quick smoke test pass `--data-mode sample` instead.

In [ ]:
!pip install -q typer rich pyyaml tqdm
!python run.py setup --data-mode all

## 3) Status — sanity check before training

In [ ]:
!python run.py status
!nvidia-smi

## 4) Production training — full STEAD on A100

Defaults from `configs/config.yaml` + `configs/model_config.yaml`:
- batch_size 128 · epochs 100 · lr 3e-4 (OneCycleLR)
- bf16 autocast · EMA decay 0.999 · grad clip 1.0
- early stop patience 12 · checkpoints mirrored to Drive every epoch

Expected wall time on A100 80 GB ≈ 4–7 h depending on shard read speed.

In [ ]:
!python run.py train --data-mode all

## 5) Resume — if Colab dropped mid-run

`--resume <exp_id>` continues writing into the same experiment directory and restores model + optimizer + scheduler + scaler + EMA shadow + RNG state from `experiments/exp_NNN/checkpoints/last.pth`.

Find the latest exp id with `python run.py status`.

In [ ]:
# Edit the id below to match the run you want to continue, then execute.
!python run.py train --resume 1 --data-mode all

## 6) Eval — latest experiment, val split

In [ ]:
!python run.py eval --split val

## 7) Inference — plot one trace from test split

In [ ]:
!python run.py infer --split test

## 8) TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir experiments --port 6006